In [1]:
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "esign-t28-qa"
NOTEBOOK_PATH = "notebooks/qa/51-esign-t28-qa.ipynb"
TOWER = 28
TOWER_NAME = "Tower of eSign + FDA Part 11"
MIN_SCORE = 70
BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"
results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""

print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

Tower 28: Tower of eSign + FDA Part 11


In [2]:
# Cell 1: AuditChain — hash-chained audit log
print("=== AuditChain (FDA 21 CFR Part 11) ===")

chain_py = read_file(SRC / "audit" / "chain.py")

# P1: AuditChain class exists
record("T28-P01", "AuditChain class exists in audit/chain.py",
       "class AuditChain" in chain_py,
       "Append-only hash-chained audit log")

# P2: AuditEntry dataclass has FDA-required fields
record("T28-P02", "AuditEntry has user_id and token_id (attributable)",
       "user_id: str" in chain_py and "token_id: str" in chain_py,
       "ALCOA-A: Attributable")

# P3: AuditEntry has timestamp
record("T28-P03", "AuditEntry has timestamp field (contemporaneous)",
       "timestamp: str" in chain_py,
       "ALCOA-C: ISO 8601 UTC")

# P4: AuditEntry has prev_hash and entry_hash
record("T28-P04", "AuditEntry has prev_hash and entry_hash for chain linking",
       "prev_hash: str" in chain_py and "entry_hash: str" in chain_py,
       "SHA-256 sequential hash chain")

# P5: AuditChain uses SHA-256
record("T28-P05", "AuditChain uses SHA-256 for hash computation",
       "hashlib.sha256" in chain_py,
       "Cryptographic hash for tamper detection")

# P6: GENESIS_HASH is all zeros (64 chars)
record("T28-P06", "GENESIS_HASH defined as 64-char zero string",
       '"0" * 64' in chain_py or "GENESIS_HASH = \"0\" * 64" in chain_py,
       "First entry links to genesis")

# P7: verify_integrity method exists
record("T28-P07", "verify_integrity method checks entire chain",
       "def verify_integrity" in chain_py,
       "Returns valid/invalid with break_at index")

=== AuditChain (FDA 21 CFR Part 11) ===
PASS: AuditChain class exists in audit/chain.py -- Append-only hash-chained audit log
PASS: AuditEntry has user_id and token_id (attributable) -- ALCOA-A: Attributable
PASS: AuditEntry has timestamp field (contemporaneous) -- ALCOA-C: ISO 8601 UTC
PASS: AuditEntry has prev_hash and entry_hash for chain linking -- SHA-256 sequential hash chain
PASS: AuditChain uses SHA-256 for hash computation -- Cryptographic hash for tamper detection
PASS: GENESIS_HASH defined as 64-char zero string -- First entry links to genesis
PASS: verify_integrity method checks entire chain -- Returns valid/invalid with break_at index


In [3]:
# Cell 2: EvidenceChainManager — two-stream evidence
print("=== EvidenceChainManager ===")

# P8: EvidenceChainManager class exists
record("T28-P08", "EvidenceChainManager class for two-stream evidence",
       "class EvidenceChainManager" in chain_py,
       "Execution chain + auth chain sharing run_id")

# P9: EvidenceChainManager has log_execution and log_auth
record("T28-P09", "EvidenceChainManager has log_execution and log_auth methods",
       "def log_execution" in chain_py and "def log_auth" in chain_py,
       "Two parallel evidence streams")

# P10: EvidenceChainManager has seal method
record("T28-P10", "EvidenceChainManager has seal method (freezes chains)",
       "def seal" in chain_py and "write_seal" in chain_py,
       "Sealed chains cannot be modified")

# P11: ChainSealedError prevents post-seal writes
record("T28-P11", "ChainSealedError raised on post-seal append",
       "class ChainSealedError" in chain_py,
       "Append after seal raises specific error")

# P12: EvidenceChainManager has validate_all
record("T28-P12", "EvidenceChainManager has validate_all for both chains",
       "def validate_all" in chain_py or "def validate_chain" in chain_py,
       "Validates execution + auth chains independently")

# P13: e_sign static method for digital signatures
record("T28-P13", "e_sign static method for computing signature hashes",
       "def e_sign" in chain_py and "sha256" in chain_py,
       "sha256(user_id + timestamp + meaning + record_hash)")

# P14: merge_cross_app for multi-application evidence
record("T28-P14", "merge_cross_app method for cross-application evidence merge",
       "def merge_cross_app" in chain_py,
       "Combines chains from multiple apps into unified view")

=== EvidenceChainManager ===
PASS: EvidenceChainManager class for two-stream evidence -- Execution chain + auth chain sharing run_id
PASS: EvidenceChainManager has log_execution and log_auth methods -- Two parallel evidence streams
PASS: EvidenceChainManager has seal method (freezes chains) -- Sealed chains cannot be modified
PASS: ChainSealedError raised on post-seal append -- Append after seal raises specific error
PASS: EvidenceChainManager has validate_all for both chains -- Validates execution + auth chains independently
PASS: e_sign static method for computing signature hashes -- sha256(user_id + timestamp + meaning + record_hash)
PASS: merge_cross_app method for cross-application evidence merge -- Combines chains from multiple apps into unified view


In [4]:
# Cell 3: ALCOA+ compliance + retention
print("=== ALCOA+ Compliance + Retention ===")

alcoa_py = read_file(SRC / "audit" / "alcoa.py")
retention_py = read_file(SRC / "audit" / "retention.py")

# P15: ALCOA+ compliance module exists
record("T28-P15", "ALCOA+ compliance validation module exists",
       "ALCOAReport" in alcoa_py,
       "9-principle compliance report")

# P16: ALCOAReport has all 9 principles
alcoa_principles = ["attributable", "legible", "contemporaneous", "original",
                    "accurate", "complete", "consistent", "enduring", "available"]
found = [p for p in alcoa_principles if f"{p}: bool" in alcoa_py or f"{p}:" in alcoa_py]
record("T28-P16", "ALCOAReport covers all 9 ALCOA+ principles",
       len(found) >= 9,
       f"Found {len(found)}/9: {found}")

# P17: Retention policy module exists
record("T28-P17", "Retention policy engine exists (FDA 11.10(c))",
       "RetentionPolicy" in retention_py and "RetentionEngine" in retention_py,
       "Record retention with min/max days")

# P18: Retention has tier-based overrides
record("T28-P18", "Retention policy has tier-based overrides",
       "tier_overrides" in retention_py and "free" in retention_py,
       "Different retention periods per subscription tier")

# P19: Retention has can_delete method (blocks deletion within window)
record("T28-P19", "Retention engine blocks deletion within retention window",
       "def can_delete" in retention_py and "allowed" in retention_py,
       "Fail-closed: records within window cannot be deleted")

# P20: Retention has records_to_archive method
record("T28-P20", "Retention engine identifies records for archival",
       "def records_to_archive" in retention_py,
       "Auto-archive past max_days (not delete)")

=== ALCOA+ Compliance + Retention ===
PASS: ALCOA+ compliance validation module exists -- 9-principle compliance report
PASS: ALCOAReport covers all 9 ALCOA+ principles -- Found 9/9: ['attributable', 'legible', 'contemporaneous', 'original', 'accurate', 'complete', 'consistent', 'enduring', 'available']
PASS: Retention policy engine exists (FDA 11.10(c)) -- Record retention with min/max days
PASS: Retention policy has tier-based overrides -- Different retention periods per subscription tier
PASS: Retention engine blocks deletion within retention window -- Fail-closed: records within window cannot be deleted
PASS: Retention engine identifies records for archival -- Auto-archive past max_days (not delete)


In [5]:
# Cell 4: OAuth3 evidence chain + AuditEntry ALCOA fields
print("=== OAuth3 Evidence + Part 11 References ===")

oauth3_evidence = read_file(SRC / "oauth3" / "evidence.py")

# P21: OAuth3 evidence chain has verify_chain method
record("T28-P21", "OAuth3 EvidenceChain has verify_chain for tamper detection",
       "def verify_chain" in oauth3_evidence,
       "Walks genesis to tail, checks hash links")

# P22: AuditEntry has meaning field (signature meaning per Part 11.50)
record("T28-P22", "AuditEntry has meaning field (Part 11.50 signature manifestation)",
       "meaning: str" in chain_py,
       'Signature meaning: authorized/reviewed/delegated')

# P23: AuditEntry has scope_used field
record("T28-P23", "AuditEntry has scope_used field (scope exercised)",
       "scope_used: str" in chain_py,
       "OAuth3 scope that was exercised")

# P24: AuditEntry has step_up_performed field
record("T28-P24", "AuditEntry has step_up_performed flag",
       "step_up_performed: bool" in chain_py,
       "Records whether re-confirmation was required")

# P25: Chain docstring references FDA 21 CFR Part 11
record("T28-P25", "Chain module references FDA 21 CFR Part 11",
       "21 CFR Part 11" in chain_py or "Part 11" in chain_py,
       "FDA compliance is explicitly referenced")

=== OAuth3 Evidence + Part 11 References ===
PASS: OAuth3 EvidenceChain has verify_chain for tamper detection -- Walks genesis to tail, checks hash links
PASS: AuditEntry has meaning field (Part 11.50 signature manifestation) -- Signature meaning: authorized/reviewed/delegated
PASS: AuditEntry has scope_used field (scope exercised) -- OAuth3 scope that was exercised
PASS: AuditEntry has step_up_performed flag -- Records whether re-confirmation was required
PASS: Chain module references FDA 21 CFR Part 11 -- FDA compliance is explicitly referenced


In [6]:
# Evidence summary cell
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"
summary = {"surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME, "timestamp": datetime.now().isoformat(), "total_probes": total, "passed": passed, "failed": failed, "score": score, "grade": grade, "finding_rate": finding_rate, "converged": finding_rate < 5, "findings": [r for r in results if r["status"] == "FAIL"], "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]}
print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))

TOWER 28: Tower of eSign + FDA Part 11
Probes: 25 | Passed: 25 | Failed: 0
Score: 100.0% | Grade: A+ | Finding rate: 0.0%
Converged: YES
Evidence hash: 89ef67d9edb312f1

{
  "surface": "esign-t28-qa",
  "tower": 28,
  "tower_name": "Tower of eSign + FDA Part 11",
  "timestamp": "2026-03-06T11:29:24.721786",
  "total_probes": 25,
  "passed": 25,
  "failed": 0,
  "score": 100.0,
  "grade": "A+",
  "finding_rate": 0.0,
  "converged": true,
  "findings": [],
  "evidence_hash": "89ef67d9edb312f1"
}
